# Phase 0: Tokenizer Round-Trip Distortion

**Thesis:** Language representations destroy the metric structure of numeric spaces.

**This experiment:** Measure pure information loss from encoding numbers as tokens and decoding back, with no model inference involved. This isolates the tokenization channel capacity for numeric information.

**Key test:** Mantel test — correlation between pairwise distance matrices before and after round-trip. If correlation < 1.0, the tokenizer destroys metric structure.

**Controls:**
- `python_str`: float64 → str() → float64 (isolates tokenization from serialization)
- `float16`: float64 → float16 → float64 (known quantization loss for calibration)

---

## Setup (Colab or fresh environment)

### One-time HuggingFace setup (for Llama-3 and Mistral tokenizers)

1. Create a HuggingFace account at [huggingface.co](https://huggingface.co) if you don't have one
2. Create an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) — select **Read** access
3. Accept the model licenses (click "Accept" on each page):
   - [meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
   - [mistralai/Mistral-7B-Instruct-v0.3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
4. Llama approval is usually instant; Mistral is instant.

**If you skip this step**, the notebook will still run — it gracefully falls back to GPT-4o (tiktoken, no auth needed) plus the two control baselines. You'll get partial but useful results.

### Then run the setup cell below

It installs dependencies, clones the repo, and prompts you to paste your HF token.

In [ ]:
# ============================================================
# Setup — run this cell first
# ============================================================
# Works on: Google Colab, local Jupyter, or any Python 3.11+ env
# No GPU needed — this experiment is CPU-only.

import os, subprocess

# Detect environment
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(globals().get("get_ipython", lambda: "")())

if IN_COLAB:
    # Install dependencies
    subprocess.check_call(["pip", "install", "-q", "transformers", "tiktoken", "scipy", "seaborn", "tqdm"])

    # Clone repo (skip if already cloned)
    if not os.path.exists("ai-interplay"):
        subprocess.check_call(["git", "clone", "https://github.com/richardmsong/ai-interplay.git"])
    os.chdir("ai-interplay")
    print("Working directory:", os.getcwd())

    # HuggingFace login for gated models (Llama-3, Mistral)
    # If you skip this, the notebook still works with GPT-4o + baselines.
    try:
        from huggingface_hub import login
        login()  # will prompt for token
    except Exception as e:
        print(f"HF login skipped: {e}")
        print("Llama-3 and Mistral tokenizers won't be available.")
        print("GPT-4o + baselines will still run fine.")
else:
    print("Local environment detected — skipping Colab setup.")
    print("Make sure you've run: pip install -e .")
    print("And if you want Llama/Mistral: huggingface-cli login")

In [ ]:
import sys
from pathlib import Path

# Handle imports from both Colab (repo root) and local (notebooks/ dir)
for candidate in [".", ".."]:
    if Path(candidate, "src", "tokenizers", "wrappers.py").exists():
        sys.path.insert(0, candidate)
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from tqdm.auto import tqdm

from src.tokenizers.wrappers import (
    get_tokenizer, get_all_tokenizer_names,
    SERIALIZATION_FORMATS, deserialize_numbers,
    Float16Baseline,
)
from src.evaluation.vectors import get_phase0_configs, VectorBatch
from src.evaluation.distortion import (
    run_full_analysis, FullDistortionReport,
    compute_element_wise, compute_vector_wise, mantel_test,
)

sns.set_theme(style="whitegrid", font_scale=1.2)
RESULTS_DIR = Path("results/phase0") if Path("results").exists() else Path("../results/phase0")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready. Results will be saved to {RESULTS_DIR.resolve()}")

## 1. Load tokenizers

We test three real tokenizers plus two controls. If any tokenizer fails to load (e.g., gated Llama model), we skip it and note it.

In [ ]:
tokenizers = {}
for name in get_all_tokenizer_names():
    try:
        tokenizers[name] = get_tokenizer(name)
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

print(f"\nLoaded {len(tokenizers)} tokenizers.")

## 2. Generate test vectors

In [ ]:
batches = get_phase0_configs(seed=42)
total_vectors = sum(b.vectors.shape[0] for b in batches)
print(f"Generated {len(batches)} batches, {total_vectors} vectors total.")
print(f"Distributions: {set(b.distribution for b in batches)}")
print(f"Dimensions: {set(b.dim for b in batches)}")
print(f"Precisions: {set(b.precision_digits for b in batches)}")

## 3. Run round-trip experiments

For each (batch, tokenizer, serialization format), we:
1. Serialize each vector to text
2. Tokenize → detokenize (round-trip)
3. Parse numbers back from the decoded text
4. Measure distortion at all three levels

In [ ]:
def run_batch_experiment(
    batch: VectorBatch,
    tokenizer_name: str,
    fmt_name: str,
) -> FullDistortionReport | None:
    """Run round-trip on a single batch with one tokenizer and format."""
    tok = tokenizers[tokenizer_name]
    serialize = SERIALIZATION_FORMATS[fmt_name]
    
    # Special handling for float16 baseline
    if tokenizer_name == "float16":
        assert isinstance(tok, Float16Baseline)
        recovered = tok.round_trip_numeric(batch.vectors)
        return run_full_analysis(
            original=batch.vectors,
            recovered=recovered,
            n_tokens_total=0,
            n_length_mismatches=0,
            tokenizer_name=tokenizer_name,
            serialization_format="native",
            distribution=batch.distribution,
            dim=batch.dim,
            precision_digits=batch.precision_digits,
            mantel_permutations=1000,  # fewer perms for speed in Phase 0
        )
    
    recovered_vecs = []
    total_tokens = 0
    length_mismatches = 0
    
    for vec in batch.vectors:
        text = serialize(vec)
        rt = tok.round_trip(text)
        total_tokens += rt.num_tokens
        
        recovered = deserialize_numbers(rt.decoded_text)
        
        # Handle length mismatches
        if len(recovered) != len(vec):
            length_mismatches += 1
            # Pad or truncate to match original length
            if len(recovered) < len(vec):
                recovered = np.pad(recovered, (0, len(vec) - len(recovered)), constant_values=np.nan)
            else:
                recovered = recovered[:len(vec)]
        
        recovered_vecs.append(recovered)
    
    recovered_arr = np.array(recovered_vecs)
    
    # Replace NaNs with 0 for metric computation (but track mismatch rate)
    recovered_clean = np.nan_to_num(recovered_arr, nan=0.0)
    
    return run_full_analysis(
        original=batch.vectors,
        recovered=recovered_clean,
        n_tokens_total=total_tokens,
        n_length_mismatches=length_mismatches,
        tokenizer_name=tokenizer_name,
        serialization_format=fmt_name,
        distribution=batch.distribution,
        dim=batch.dim,
        precision_digits=batch.precision_digits,
        mantel_permutations=1000,
    )

In [ ]:
# Run the full experiment matrix
# For float16 baseline, only run once (no serialization format applies)
# For real tokenizers, test all serialization formats

results: list[FullDistortionReport] = []

real_tokenizer_names = [n for n in tokenizers if n != "float16"]
fmt_names = list(SERIALIZATION_FORMATS.keys())

# Count total experiments for progress bar
n_experiments = len(batches) * (
    len(real_tokenizer_names) * len(fmt_names) + (1 if "float16" in tokenizers else 0)
)
print(f"Running {n_experiments} experiments...")

with tqdm(total=n_experiments) as pbar:
    for batch in batches:
        # Float16 baseline (format-independent)
        if "float16" in tokenizers:
            report = run_batch_experiment(batch, "float16", "csv")
            if report:
                results.append(report)
            pbar.update(1)
        
        # Real tokenizers x formats
        for tok_name, fmt_name in product(real_tokenizer_names, fmt_names):
            report = run_batch_experiment(batch, tok_name, fmt_name)
            if report:
                results.append(report)
            pbar.update(1)

print(f"\nCompleted {len(results)} experiments.")

## 4. Compile results into a DataFrame

In [ ]:
rows = []
for r in results:
    row = {
        "tokenizer": r.tokenizer_name,
        "format": r.serialization_format,
        "distribution": r.distribution,
        "dim": r.dim,
        "precision_digits": r.precision_digits,
        "n_vectors": r.n_vectors,
        "n_tokens_total": r.n_tokens_total,
        "length_mismatch_rate": r.length_mismatch_rate,
        # Element-wise
        "mean_abs_error": r.element_wise.mean_abs_error,
        "median_abs_error": r.element_wise.median_abs_error,
        "mean_rel_error": r.element_wise.mean_rel_error,
        "median_rel_error": r.element_wise.median_rel_error,
        "mean_sig_digits": r.element_wise.mean_sig_digits,
        "perfect_recovery_rate": r.element_wise.perfect_recovery_rate,
        # Vector-wise
        "mean_normalized_l2": r.vector_wise.mean_normalized_l2,
        "mean_cosine_sim": r.vector_wise.mean_cosine_similarity,
        "mean_spearman_rho": r.vector_wise.mean_spearman_rho,
        # Mantel
        "mantel_r": r.mantel.correlation if r.mantel else None,
        "mantel_p": r.mantel.p_value if r.mantel else None,
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "round_trip_results.csv", index=False)
print(f"Results: {len(df)} rows, saved to {RESULTS_DIR / 'round_trip_results.csv'}")
df.head(10)

## 5. Analysis: Element-wise distortion

First question: how much raw error does each tokenizer introduce?

In [ ]:
# Aggregate: mean significant digits preserved, by tokenizer and format
agg = (
    df[df["tokenizer"] != "float16"]
    .groupby(["tokenizer", "format"])
    .agg(
        mean_sig_digits=("mean_sig_digits", "mean"),
        perfect_recovery=("perfect_recovery_rate", "mean"),
        length_mismatch=("length_mismatch_rate", "mean"),
    )
    .round(3)
    .sort_values("mean_sig_digits", ascending=False)
)
print("Significant digits preserved (higher is better):")
agg

In [ ]:
# Plot: significant digits by tokenizer, faceted by format
plot_df = df[~df["tokenizer"].isin(["float16"])].copy()
plot_df["precision_label"] = plot_df["precision_digits"].fillna("full").astype(str)

fig, axes = plt.subplots(1, len(fmt_names), figsize=(5 * len(fmt_names), 5), sharey=True)
if len(fmt_names) == 1:
    axes = [axes]

for ax, fmt in zip(axes, fmt_names):
    subset = plot_df[plot_df["format"] == fmt]
    sns.boxplot(data=subset, x="tokenizer", y="mean_sig_digits", ax=ax)
    ax.set_title(f"Format: {fmt}")
    ax.set_ylabel("Mean significant digits" if fmt == fmt_names[0] else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Significant Digits Preserved After Tokenizer Round-Trip", y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "sig_digits_by_tokenizer.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Analysis: Metric structure preservation (Mantel test)

**This is the key result.** Does the tokenizer preserve the distance relationships between vectors?

In [ ]:
# Filter to experiments where Mantel test was computed (dim > 1)
mantel_df = df[df["mantel_r"].notna()].copy()

print(f"Mantel test results: {len(mantel_df)} experiments (dim > 1)")
print()

# Summary by tokenizer
mantel_summary = (
    mantel_df
    .groupby("tokenizer")
    .agg(
        mean_mantel_r=("mantel_r", "mean"),
        min_mantel_r=("mantel_r", "min"),
        pct_significant=("mantel_p", lambda x: (x < 0.05).mean()),
    )
    .round(4)
    .sort_values("mean_mantel_r", ascending=False)
)
print("Mantel test summary by tokenizer:")
print("(mean_mantel_r = 1.0 means perfect metric structure preservation)")
print()
mantel_summary

In [ ]:
# Plot: Mantel correlation by tokenizer and format
fig, ax = plt.subplots(figsize=(10, 6))

mantel_plot = mantel_df[~mantel_df["tokenizer"].isin(["float16"])]
sns.boxplot(
    data=mantel_plot, x="tokenizer", y="mantel_r", hue="format", ax=ax
)
ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, label="Perfect preservation")

# Add float16 baseline as horizontal band
f16 = mantel_df[mantel_df["tokenizer"] == "float16"]["mantel_r"]
if len(f16) > 0:
    ax.axhspan(f16.min(), f16.max(), alpha=0.1, color="green", label=f"float16 baseline ({f16.mean():.4f})")

ax.set_ylabel("Mantel correlation (1.0 = perfect)")
ax.set_xlabel("")
ax.set_title("Metric Structure Preservation: Mantel Test")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "mantel_by_tokenizer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Drill down: Mantel r by magnitude and precision
# (Where does metric structure break down?)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By distribution (proxy for magnitude)
sns.boxplot(
    data=mantel_plot, x="distribution", y="mantel_r", hue="tokenizer", ax=axes[0]
)
axes[0].axhline(y=1.0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Mantel r by Distribution")
axes[0].set_ylabel("Mantel correlation")
axes[0].tick_params(axis="x", rotation=45)

# By precision
prec_plot = mantel_plot.copy()
prec_plot["precision_label"] = prec_plot["precision_digits"].fillna("full").astype(str)
sns.boxplot(
    data=prec_plot, x="precision_label", y="mantel_r", hue="tokenizer", ax=axes[1],
    order=["0", "2", "4", "8", "full"],
)
axes[1].axhline(y=1.0, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("Mantel r by Precision")
axes[1].set_ylabel("")
axes[1].set_xlabel("Decimal places")

fig.suptitle("Where Does Metric Structure Break Down?", y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "mantel_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analysis: Token efficiency

How many tokens does each format require? More tokens = more inference cost in Phase 1+.

In [ ]:
token_df = df[~df["tokenizer"].isin(["float16"])].copy()
token_df["tokens_per_element"] = token_df["n_tokens_total"] / (token_df["n_vectors"] * token_df["dim"])

token_summary = (
    token_df
    .groupby(["tokenizer", "format"])
    .agg(mean_tokens_per_element=("tokens_per_element", "mean"))
    .round(2)
    .sort_values("mean_tokens_per_element")
)
print("Tokens per numeric element (lower = more efficient):")
token_summary

## 8. Key findings summary

In [ ]:
print("=" * 60)
print("PHASE 0 SUMMARY: Tokenizer Round-Trip Distortion")
print("=" * 60)
print()

# Overall perfect recovery rate
for tok_name in sorted(df["tokenizer"].unique()):
    subset = df[df["tokenizer"] == tok_name]
    pr = subset["perfect_recovery_rate"].mean()
    sd = subset["mean_sig_digits"].mean()
    mr = subset["mantel_r"].mean() if subset["mantel_r"].notna().any() else float("nan")
    print(f"{tok_name:15s}  perfect_recovery={pr:.3f}  sig_digits={sd:.1f}  mantel_r={mr:.4f}")

print()
print("Interpretation:")
print("- perfect_recovery: fraction of numbers recovered exactly (1.0 = lossless)")
print("- sig_digits: mean significant digits preserved (16 = float64 max)")
print("- mantel_r: metric structure preservation (1.0 = perfect, <1.0 = distortion)")
print()

# Decision gate
real_toks = mantel_df[~mantel_df["tokenizer"].isin(["float16", "python_str"])]
if len(real_toks) > 0:
    worst_mantel = real_toks["mantel_r"].min()
    mean_mantel = real_toks["mantel_r"].mean()
    print(f"Worst Mantel r across real tokenizers: {worst_mantel:.4f}")
    print(f"Mean Mantel r across real tokenizers:  {mean_mantel:.4f}")
    print()
    if mean_mantel > 0.999:
        print("→ Tokenizers preserve metric structure well.")
        print("  The translation tax may be minimal at the tokenization level.")
        print("  Proceed to Phase 1 to test whether the MODEL introduces distortion.")
    elif mean_mantel > 0.99:
        print("→ Small but measurable metric structure distortion.")
        print("  Investigate which conditions cause the most distortion (magnitude? precision?).")
        print("  Proceed to Phase 1 — the distortion may compound across agent hops.")
    else:
        print("→ Significant metric structure distortion detected!")
        print("  This directly supports the thesis at the tokenization level.")
        print("  Characterize the failure modes before proceeding.")